# Day 024 Project Solution — Web Page → Structured JSON

A `PageExtractor` that fetches any URL, strips HTML to plain text, uses the LLM to fill a Pydantic schema, validates the result, and returns structured data.

In [ ]:
import re
import json
import requests
import ollama
from bs4 import BeautifulSoup
from pydantic import BaseModel


def clean_html_text(html_string: str) -> str:
    soup = BeautifulSoup(html_string, "html.parser")
    for tag in soup(["script", "style"]):
        tag.decompose()
    text = soup.get_text(separator="\n", strip=True)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def extract_schema_fields(
    text: str, fields: list[str], model: str = "llama3.2"
) -> dict:
    fields_json = json.dumps(fields)
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a structured data extractor. "
                    f"Extract the following fields from the text: {fields_json}. "
                    "Return JSON with exactly these keys. "
                    "Use null for any field you cannot find. "
                    "Return only valid JSON, no explanation."
                ),
            },
            {
                "role": "user",
                "content": f"Extract from this text:\n\n{text[:3000]}",
            },
        ],
        format="json",
    )
    raw = response["message"]["content"]
    try:
        return json.loads(raw)
    except Exception:
        return {f: None for f in fields}


def validate_extracted(raw: dict, model_class: type[BaseModel]) -> BaseModel | None:
    try:
        return model_class.model_validate(raw)
    except Exception:
        return None


def scrape_and_extract(
    url: str, fields: list[str], model: str = "llama3.2"
) -> dict:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    text = clean_html_text(response.text)
    return extract_schema_fields(text, fields, model=model)


def batch_scrape_extract(
    urls: list[str],
    fields: list[str],
    model: str = "llama3.2",
) -> list[dict]:
    results = []
    for url in urls:
        try:
            data = scrape_and_extract(url, fields, model=model)
            results.append({"url": url, "status": "ok", "data": data})
        except Exception as e:
            results.append({"url": url, "status": "error", "error": str(e)})
    return results


class PageExtractor:
    def __init__(self, model_class: type[BaseModel], model: str = "llama3.2"):
        self.model_class = model_class
        self.llm_model = model
        self.fields = list(model_class.model_fields.keys())

    def extract(self, url: str) -> BaseModel | None:
        data = scrape_and_extract(url, self.fields, model=self.llm_model)
        return validate_extracted(data, self.model_class)

    def extract_many(self, urls: list[str]) -> list[dict]:
        return batch_scrape_extract(urls, self.fields, model=self.llm_model)

    def to_json(self, instance: BaseModel) -> str:
        return json.dumps(instance.model_dump(), indent=2)


class SiteInfo(BaseModel):
    site_name: str | None = None
    main_purpose: str | None = None

## Action 1 — Extract Structured Info from books.toscrape.com

In [ ]:
extractor = PageExtractor(SiteInfo)
info = extractor.extract("https://books.toscrape.com")
print("Extracted site info:")
if info:
    print(f"  site_name   : {info.site_name}")
    print(f"  main_purpose: {info.main_purpose}")
    print("\nJSON output:")
    print(extractor.to_json(info))
else:
    print("  (extraction returned None — LLM validation failed)")
    fallback = SiteInfo(site_name="Books to Scrape", main_purpose="Scraping practice")
    print(extractor.to_json(fallback))

## Action 2 — Batch Extract: One Good URL, One Bad URL

In [ ]:
urls = ["https://books.toscrape.com", "http://localhost:9999/"]
results = extractor.extract_many(urls)
print("\nBatch results:")
for r in results:
    label = r["url"][:45]
    if r["status"] == "ok":
        fields_extracted = list(r["data"].keys())
        print(f"  {label} → ok (fields: {fields_extracted})")
    else:
        print(f"  {label} → error: {str(r['error'])[:50]}")

## Action 3 — Validate and Serialize

In [ ]:
# Show validate_extracted in action
raw_good = {'site_name': 'Books to Scrape', 'main_purpose': 'Practice scraping'}
raw_bad  = {'wrong_key': 'ignored'}

validated = validate_extracted(raw_good, SiteInfo)
rejected  = validate_extracted(raw_bad, SiteInfo)

print('\nValidation demo:')
print(f'  raw_good → {type(validated).__name__}: {validated}')
print(f'  raw_bad  → {rejected!r} (None = validation returned empty model or failed)')
print('\nExtraction complete!')